# Dropout

**Capítulo 3 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_multilayer-perceptrons/dropout.ipynb` · [Lección original](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Dropout
<a id="sec_dropout"></a>

Pensemos brevemente en lo que esperamos de un buen modelo predictivo. Queremos que se forme bien en datos invisibles. La teoría de la generalización clásica sugiere que para cerrar la brecha entre el tren y el rendimiento de la prueba, debemos aspirar a un modelo simple. La simplicidad puede venir en forma de un pequeño número de dimensiones. Lo exploramos al discutir las funciones de base monomial de los modelos lineales en [Referencia sec_generalization_basics](https://d2l.ai/chapter_linear-regression/generalization.html#sec-generalization-basics). Además, como vimos al discutir la decaimiento de pesos (regularización $\ell_2$) en [Referencia sec_weight_decay](https://d2l.ai/chapter_linear-regression/weight-decay.html#sec-weight-decay), la norma (inversa) de los parámetros también representa una medida útil de simplicidad. Otra noción útil de simplicidad es la suavidad, es decir, que la función no debe ser sensible a los pequeños cambios a sus entradas. Por ejemplo, cuando clasificamos imágenes, esperaríamos que añadir algo de ruido aleatorio a los píxeles debería ser principalmente inofensivo.

[Bishop.1995](https://d2l.ai/chapter_references/zreferences.html) formalizó
esta idea cuando demostró que el entrenamiento con el ruido de entrada es equivalente a la regularización Tikhonov. Este trabajo dibujó una clara conexión matemática entre el requisito de que una función sea suave (y por lo tanto simple), y el requisito de que sea resistente a las perturbaciones en la entrada.

Luego, [Srivastava.Hinton.Krizhevsky.ea.2014](https://d2l.ai/chapter_references/zreferences.html) desarrolló una idea inteligente para cómo aplicar la idea de Bishop a las capas internas de una red, también. Su idea, llamada *dropout*, implica inyectar ruido mientras se computa cada capa interna durante la propagación hacia delante, y se ha convertido en una técnica estándar para entrenar redes neuronales. El método se llama *dropout* porque literalmente *desechamos* algunas neuronas durante el entrenamiento. A lo largo del entrenamiento, en cada iteración, la dropout estándar consiste en cero una fracción de los nodos en cada capa antes de calcular la capa posterior.

Para ser claros, estamos imponiendo nuestra propia narrativa con el vínculo con Bishop. El artículo original sobre dropout ofrece intuición a través de una sorprendente analogía con la reproducción sexual. Los autores argumentan que la sobreadaptación de la red neuronal se caracteriza por un estado en el que cada capa se basa en un patrón específico de activaciones en la capa anterior, llamando a esta condición *co-adaptación*. El dropout, afirman, rompe la coadaptación tal como se argumenta que la reproducción sexual rompe genes coadaptados. Si bien tal justificación de esta teoría es ciertamente para el debate, la técnica de dropout en sí ha demostrado ser duradera, y varias formas de dropout se implementan en la mayoría de las bibliotecas de aprendizaje profundo.

El reto clave es cómo inyectar este ruido. Una idea es inyectarlo de una manera *insensata* para que el valor esperado de cada capa---mientras que la fijación de los otros---equivale al valor que habría tomado ruido ausente. En el trabajo de Bishop, añadió el ruido gaussiano a las entradas a un modelo lineal. En cada iteración de entrenamiento, añadió el ruido muestreado de una distribución con media cero $\epsilon \sim \mathcal{N}(0,\sigma^2)$ a la entrada $\mathbf{x}$, dando un punto perturbado $\mathbf{x}' = \mathbf{x} + \epsilon$. En expectativa, $E[\mathbf{x}'] = \mathbf{x}$.

En la regularización de dropout estándar, uno ceros hacia fuera alguna fracción de los nodos en cada capa y luego *debías* cada capa normalizando por la fracción de nodos que fueron retenidos (no abandonado). En otras palabras, con *probabilidad de dropout* $p$, cada activación intermedia $h$ es reemplazada por una variable aleatoria $h'$ como sigue:

$$
\begin{aligned}
h' =
\begin{cases}
    0 & \textrm{ with probability } p \\
    \frac{h}{1-p} & \textrm{ otherwise}
\end{cases}
\end{aligned}
$$

Por diseño, la expectativa se mantiene sin cambios, es decir, $E[h'] = h$.


In [ ]:
import torch
from torch import nn
from laboratorio import d2l

## Dropout en la práctica
Recordemos el MLP con una capa oculta y cinco unidades ocultas de [Referencia fig_mlp](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html#fig-mlp). Cuando aplicamos el dropout a una capa oculta, cero hacia fuera cada unidad oculta con probabilidad $p$, el resultado puede ser visto como una red que contiene sólo un subconjunto de las neuronas originales. En [Referencia fig_dropout2](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html#fig-dropout2), $h_2$ y $h_5$ se eliminan. Por lo tanto, el cálculo de las salidas ya no depende de $h_2$ o $h_5$ y su gradiente respectivo también desaparece al realizar la retropropagación. De esta manera, el cálculo de la capa de salida no puede ser demasiado dependiente de cualquier elemento de $h_1, \ldots, h_5$.

![MLP antes y después de aplicar dropout.](../recursos/originales/dropout2.svg)
<a id="fig_dropout2"></a>

Típicamente, deshabilitamos la dropout durante la evaluación. Dado un modelo entrenado y un nuevo ejemplo, no eliminamos ningún nodo y por lo tanto no necesitamos normalizar. Sin embargo, hay algunas excepciones: algunos investigadores usan la dropout durante la evaluación como un heurístico para estimar la *incertidumbre* de las predicciones de la red neuronal: si las predicciones concuerdan entre muchas salidas de dropout diferentes, entonces podríamos decir que la red es más segura.

## Implementación desde cero
Para implementar la función de dropout para una sola capa, debemos extraer tantas muestras de una variable aleatoria Bernoulli (binaria) como nuestra capa tenga dimensiones, donde la variable aleatoria toma el valor $1$ (mantener) con probabilidad $1-p$ y $0$ (gota) con probabilidad $p$. Una manera fácil de implementar esto es primero extraer muestras de la distribución uniforme $U[0, 1]$. Entonces podemos mantener aquellos nodos para los cuales la muestra correspondiente es mayor que $p$, dejando caer el resto.

En el siguiente código, ** implementamos una función `dropout_layer` que elimina los elementos en la entrada de tensor `X` con probabilidad `dropout`**, reescalando el resto como se describe anteriormente: dividiendo los sobrevivientes por `1.0-dropout`.


In [ ]:
def dropout_layer(X, dropout):
    assert 0 <= dropout <= 1
    if dropout == 1: return torch.zeros_like(X)
    mask = (torch.rand(X.shape) > dropout).float()
    return mask * X / (1.0 - dropout)

Podemos ** probar la función `dropout_layer` en algunos ejemplos**. En las siguientes líneas de código, pasamos nuestra entrada `X` a través de la operación de dropout, con probabilidades 0, 0.5 y 1, respectivamente.


In [ ]:
X = torch.arange(16, dtype = torch.float32).reshape((2, 8))
print('dropout_p = 0:', dropout_layer(X, 0))
print('dropout_p = 0.5:', dropout_layer(X, 0.5))
print('dropout_p = 1:', dropout_layer(X, 1))

### Definir el modelo
El siguiente modelo aplica el dropout a la salida de cada capa oculta (después de la función de activación). Podemos establecer las probabilidades de dropout para cada capa por separado. Una opción común es establecer una menor probabilidad de dropout más cerca de la capa de entrada. Nos aseguramos de que el dropout sólo es activo durante el entrenamiento.


In [ ]:
class DropoutMLPScratch(d2l.Classifier):
    def __init__(self, num_outputs, num_hiddens_1, num_hiddens_2,
                 dropout_1, dropout_2, lr):
        super().__init__()
        self.save_hyperparameters()
        self.lin1 = nn.LazyLinear(num_hiddens_1)
        self.lin2 = nn.LazyLinear(num_hiddens_2)
        self.lin3 = nn.LazyLinear(num_outputs)
        self.relu = nn.ReLU()

    def forward(self, X):
        H1 = self.relu(self.lin1(X.reshape((X.shape[0], -1))))
        if self.training:
            H1 = dropout_layer(H1, self.dropout_1)
        H2 = self.relu(self.lin2(H1))
        if self.training:
            H2 = dropout_layer(H2, self.dropout_2)
        return self.lin3(H2)

### Nota docente de Hespérides

Para comparar modelos, conserva la partición y empareja las semillas. Selecciona hiperparámetros con validación y reserva el test para el final. La versión D2L de Fashion-MNIST llama «val» al test oficial: en estos derivados se separa validación del entrenamiento oficial. El modo rápido demuestra mecanismos; no permite extraer una clasificación definitiva de técnicas.

Vínculo con los apuntes: sesión 3, «Dropout».


### Entrenamiento

Lo que sigue es similar a la formación de los PPM descrita anteriormente.


In [ ]:
hparams = {'num_outputs':10, 'num_hiddens_1':256, 'num_hiddens_2':256,
           'dropout_1':0.5, 'dropout_2':0.5, 'lr':0.1}
model = DropoutMLPScratch(**hparams)
data = d2l.FashionMNIST(batch_size=256)
trainer = d2l.Trainer(max_epochs=10)
trainer.fit(model, data)

## Implementación concisa

Con API de alto nivel, todo lo que tenemos que hacer es añadir una capa `Dropout` después de cada capa totalmente conectada, pasando en la probabilidad de dropout como el único argumento a su constructor. Durante el entrenamiento, la capa `Dropout` abandonará aleatoriamente las salidas de la capa anterior (o equivalentemente, las entradas a la capa posterior) de acuerdo con la probabilidad de dropout especificada. Cuando no está en modo de entrenamiento, la capa `Dropout` simplemente pasa los datos durante la prueba.


In [ ]:
class DropoutMLP(d2l.Classifier):
    def __init__(self, num_outputs, num_hiddens_1, num_hiddens_2,
                 dropout_1, dropout_2, lr):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.Flatten(), nn.LazyLinear(num_hiddens_1), nn.ReLU(),
            nn.Dropout(dropout_1), nn.LazyLinear(num_hiddens_2), nn.ReLU(),
            nn.Dropout(dropout_2), nn.LazyLinear(num_outputs))

A continuación, **entrenamos el modelo**.


In [ ]:
model = DropoutMLP(**hparams)
trainer.fit(model, data)

## Resumen
Más allá de controlar el número de dimensiones y el tamaño del vector de peso, el dropout es otra herramienta para evitar el exceso de ajuste. A menudo las herramientas se utilizan conjuntamente. Tenga en cuenta que el dropout se utiliza sólo durante el entrenamiento: reemplaza una activación $h$ por una variable aleatoria con el valor esperado $h$.

## Ejercicios
1. ¿Qué sucede si cambias las probabilidades de dropout para las capas primera y segunda? En particular, ¿qué sucede si cambias las de ambas capas? Diseña un experimento para responder a estas preguntas, describe tus resultados cuantitativamente, y resume los resultados cualitativos.
1. Aumentar el número de épocas y comparar los resultados obtenidos cuando se utiliza dropout con aquellos cuando no se utiliza.
1. ¿Cuál es la varianza de las activaciones en cada capa oculta cuando se aplica y no se aplica el dropout? Dibuje un gráfico para mostrar cómo esta cantidad evoluciona con el tiempo para ambos modelos.
1. ¿Por qué la dropout no se utiliza típicamente en el momento de la prueba?
1. Usando el modelo de esta sección como ejemplo, compare los efectos del uso de la dropout y el deterioro del peso. ¿Qué sucede cuando la dropout y el deterioro del peso se usan al mismo tiempo? ¿Son los resultados aditivos? ¿Hay rendimientos disminuidos (o peores)? ¿Se cancelan mutuamente?
1. ¿Qué sucede si aplicamos el dropout a los pesos individuales de la matriz de peso en lugar de las activaciones?
1. Invente otra técnica para inyectar ruido aleatorio en cada capa que sea diferente de la técnica estándar de dropout. ¿Puede desarrollar un método que supere el dropout en el conjunto de datos de Fashion-MNIST (para una arquitectura fija)?


[Debate del original](https://discuss.d2l.ai/t/101)
